In [98]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False

In [99]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)
print("FAISS доступен:", FAISS_AVAILABLE)

Устройство для работы: cpu
FAISS доступен: True


In [100]:
import os
import pandas as pd

def open_document(doc_id: str, title: str):
    with open(title, 'r', encoding='utf-8') as document:
        text = document.read()
    return {
        "doc_id": doc_id,
        "title": (title[10:-4]),
        "text": text
    }

documents = []
for i, filename in enumerate(os.listdir("documents"), 1):
    doc_id = f"doc_{i:02d}"
    documents.append(open_document(doc_id, os.path.join("documents", filename)))


documents_df = pd.DataFrame(documents)
display(documents_df[["doc_id", "title","text"[:10]]])

benchmark_queries: List[Dict[str, object]] = [
    {
        "query_id": "q01",
        "query": "а какие форматы у вордовских файлов? docx это а у шаблонов что",
        "relevant_doc_ids": ["doc_03"],
    },
    {
        "query_id": "q02",
        "query": "не могу найти куда в ворде нажать чтобы кнопку добавить на панель",
        "relevant_doc_ids": ["doc_03"],
    },
    {
        "query_id": "q03",
        "query": "логином комьюнити это бесплатно? зачем оно вообще нужно",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q04",
        "query": "как в логиноме создать новый сценарий и узлы туда добавить",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q05",
        "query": "узел красным горит при запуске это ошибка? а зеленый норм",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q06",
        "query": "в экселе есть специальная вставка там кнопка транспонировать зачем она",
        "relevant_doc_ids": ["doc_01"],
    },
    {
        "query_id": "q07",
        "query": "как ячейку в экселе поменять если уже написал туда текст",
        "relevant_doc_ids": ["doc_01"],
    },
    {
        "query_id": "q08",
        "query": "панель быстрого доступа в экселе где находится и как ее настроить",
        "relevant_doc_ids": ["doc_01"],
    },
    {
        "query_id": "q09",
        "query": "можно ли в экселе строки столбцами поменять местами",
        "relevant_doc_ids": ["doc_01"],
    }]
benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,doc_id,title,text
0,doc_01,руководство ecxel,Панель быстрого доступа можно легко изменять и...
1,doc_02,Руководство Loginom,Loginom – Low-code платформа для реализации вс...
2,doc_03,руководство word,Окно программы\nMiсrosoft Word 2000 – текстовы...


,query_id,query,relevant_doc_ids
0,q01,а какие форматы у вордовских файлов? docx это ...,[doc_03]
1,q02,не могу найти куда в ворде нажать чтобы кнопку...,[doc_03]
2,q03,логином комьюнити это бесплатно? зачем оно воо...,[doc_02]
3,q04,как в логиноме создать новый сценарий и узлы т...,[doc_02]
4,q05,узел красным горит при запуске это ошибка? а з...,[doc_02]
5,q06,в экселе есть специальная вставка там кнопка т...,[doc_01]
6,q07,как ячейку в экселе поменять если уже написал ...,[doc_01]
7,q08,панель быстрого доступа в экселе где находится...,[doc_01]
8,q09,можно ли в экселе строки столбцами поменять ме...,[doc_01]


Загруженные документы представляют собой фрагменты из руководств пользователя по 3 программам: loginom, excel и word. По выбранной предметной области размуно строить retrieval / mini-RAG, потому что к руководствам чаще всего обращаются с конкретыми вопросами, в них легче всего предположить и найти конкретный ответ по ключевым словам.

In [101]:
# В этом разделе собираем self-contained реализацию:
# чанкинг -> векторизация -> индекс -> поиск -> оценка.

def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks


class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


def select_backend(device: str = "cpu") -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=device,
        )
        print("Используем dense-модель эмбеддингов.")
        return backend
    except Exception as e:
        print("Dense-модель недоступна, переключаемся на fallback.")
        print("Причина:", repr(e))
        return TfidfBackend()


def build_chunks(
    docs: List[Dict[str, str]],
    chunk_size: int,
    overlap: int,
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_idx, chunk_text_value in enumerate(parts):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                    "chunk_idx": chunk_idx,
                    "chunk_text": chunk_text_value,
                }
            )
    return rows


@dataclass
class RetrievalArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object


def build_retriever(
    docs: List[Dict[str, str]],
    chunk_size: int = 40,
    overlap: int = 10,
    device: str = "cpu",
) -> RetrievalArtifacts:
    chunks = build_chunks(docs, chunk_size=chunk_size, overlap=overlap)
    chunks_df = pd.DataFrame(chunks)

    backend = select_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if not FAISS_AVAILABLE:
        raise RuntimeError("FAISS недоступен. Для этого ноутбука ожидается установленный faiss-cpu.")

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrievalArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )


def search_chunks(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [102]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_AVAILABLE:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices
    


## Чанки / векоризация

In [9]:
def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 40,
    overlap: int = 10,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)

chunks_df = build_chunks_dataframe(documents, chunk_size=40, overlap=10)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 115


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,руководство ecxel,0,Панель быстрого доступа можно легко изменять и...,40
1,doc_01,руководство ecxel,1,направленной вниз. Линейки прокрутки Линейки п...,40
2,doc_01,руководство ecxel,2,"позволяет определить, в каком месте документа ...",40
3,doc_01,руководство ecxel,3,"состоит из рабочих листов, имена которых (Лист...",40
4,doc_01,руководство ecxel,4,Для прокручивания ярлыков используются кнопки ...,40
5,doc_01,руководство ecxel,5,"лист представляет собой таблицу, состоящую из ...",40
6,doc_01,руководство ecxel,6,"имени строки и имени столбца. Например, если я...",40
7,doc_01,руководство ecxel,7,необходимо сделать ее активной и ввести данные...,40
8,doc_01,руководство ecxel,8,Редактирование данных в ячейке можно выполнить...,40
9,doc_01,руководство ecxel,9,ячейку активной. Сделать щелчок в строке форму...,40


Для документов выбран размер чанка в 40 символов, чтобы сохранить связность текста между собой, и так же чтобы повысить уверенность модели и дать возможность более точно выбирать необходимые фрагменты текста. Перекрытие выбрано в 10 символов, те 25% от чанка, чтобы важная информация на границе чанков не терялась

## FAISS

In [ ]:
def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")

# Строим векторные представления для всех чанков.
embedder = build_embedding_backend(device=DEVICE)
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4264.94it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Форма матрицы эмбеддингов: (115, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


In [27]:
search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)


def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)


for current_query in benchmark_queries:
    display(Markdown(f"### Запрос: `{current_query["query"]}`"))
    display(search_similar_chunks(current_query["query"], top_k=3))

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


### Запрос: `а какие форматы у вордовских файлов? docx это а у шаблонов что`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_03,руководство word,19,0.5979,"поле Папка выбирается диск, на котором находит..."
1,2,doc_02,Руководство Loginom,15,0.5173,"из четырёх групп объектов: Переменные, Ссылки,..."
2,3,doc_03,руководство word,1,0.4763,готовый к печати документ без необходимости ра...


### Запрос: `не могу найти куда в ворде нажать чтобы кнопку добавить на панель`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_03,руководство word,5,0.6901,"команды, доступные в меню. Для вызова команды,..."
1,2,doc_03,руководство word,7,0.6677,имя нужной панели. Если панель присутствует на...
2,3,doc_03,руководство word,8,0.6632,которые были использованы последними. Если наж...


...

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_03,руководство word,1,0.5386,готовый к печати документ без необходимости ра...
1,2,doc_01,руководство ecxel,3,0.5297,"состоит из рабочих листов, имена которых (Лист..."
2,3,doc_01,руководство ecxel,52,0.5206,2. В группе Ячейки ленты Главная щелкнуть по с...


### Запрос: `панель быстрого доступа в экселе где находится и как ее настроить`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_01,руководство ecxel,0,0.7120,Панель быстрого доступа можно легко изменять и...
1,2,doc_02,Руководство Loginom,3,0.5539,разделено на 3 основных блока: 1. Боковое выдв...
2,3,doc_02,Руководство Loginom,17,0.5153,"источников, приемников данных, к которым можно..."


### Запрос: `можно ли в экселе строки столбцами поменять местами`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_01,руководство ecxel,31,0.6605,выделенными строками. Если выделить несколько ...
1,2,doc_01,руководство ecxel,28,0.6548,или строки. Для выделения нескольких столбцов ...
2,3,doc_01,руководство ecxel,4,0.6150,Для прокручивания ярлыков используются кнопки ...


## Retrieval

In [103]:

# Собираем baseline-конфигурацию retriever.
baseline_chunk_size = 40
baseline_overlap = 10

artifacts = build_retriever(
    documents,
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

for query in benchmark_df["query"]:
    display(Markdown(f"### Запрос: {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4806.24it/s]


Используем dense-модель эмбеддингов.
Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 115


,doc_id,title,chunk_id,chunk_idx,chunk_text
0,doc_01,руководство ecxel,doc_01_chunk_0,0,Панель быстрого доступа можно легко изменять и...
1,doc_01,руководство ecxel,doc_01_chunk_1,1,направленной вниз. Линейки прокрутки Линейки п...
2,doc_01,руководство ecxel,doc_01_chunk_2,2,"позволяет определить, в каком месте документа ..."
3,doc_01,руководство ecxel,doc_01_chunk_3,3,"состоит из рабочих листов, имена которых (Лист..."
4,doc_01,руководство ecxel,doc_01_chunk_4,4,Для прокручивания ярлыков используются кнопки ...


### Запрос: а какие форматы у вордовских файлов? docx это а у шаблонов что

,rank,score,doc_id,title,chunk_text
0,1,0.597941,doc_03,руководство word,"поле Папка выбирается диск, на котором находит..."
1,2,0.517265,doc_02,Руководство Loginom,"из четырёх групп объектов: Переменные, Ссылки,..."
2,3,0.476289,doc_03,руководство word,готовый к печати документ без необходимости ра...


...

,rank,score,doc_id,title,chunk_text
0,1,0.538628,doc_03,руководство word,готовый к печати документ без необходимости ра...
1,2,0.529739,doc_01,руководство ecxel,"состоит из рабочих листов, имена которых (Лист..."
2,3,0.520580,doc_01,руководство ecxel,2. В группе Ячейки ленты Главная щелкнуть по с...


### Запрос: панель быстрого доступа в экселе где находится и как ее настроить

,rank,score,doc_id,title,chunk_text
0,1,0.712041,doc_01,руководство ecxel,Панель быстрого доступа можно легко изменять и...
1,2,0.553890,doc_02,Руководство Loginom,разделено на 3 основных блока: 1. Боковое выдв...
2,3,0.515284,doc_02,Руководство Loginom,"источников, приемников данных, к которым можно..."


### Запрос: можно ли в экселе строки столбцами поменять местами

,rank,score,doc_id,title,chunk_text
0,1,0.660548,doc_01,руководство ecxel,выделенными строками. Если выделить несколько ...
1,2,0.654823,doc_01,руководство ecxel,или строки. Для выделения нескольких столбцов ...
2,3,0.615014,doc_01,руководство ecxel,Для прокручивания ярлыков используются кнопки ...


In [104]:
results = []
baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)


display(summary_k3)

,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,MRR@3,first_relevant_rank
0,q01,а какие форматы у вордовских файлов? docx это ...,doc_03,"doc_03, doc_02",1,1.0,1.0,1.0
1,q02,не могу найти куда в ворде нажать чтобы кнопку...,doc_03,doc_03,1,1.0,1.0,1.0
2,q03,логином комьюнити это бесплатно? зачем оно воо...,doc_02,"doc_03, doc_01",0,0.0,0.0,NaN
3,q04,как в логиноме создать новый сценарий и узлы т...,doc_02,doc_02,1,1.0,1.0,1.0
4,q05,узел красным горит при запуске это ошибка? а з...,doc_02,"doc_02, doc_01",1,1.0,1.0,1.0
5,q06,в экселе есть специальная вставка там кнопка т...,doc_01,"doc_01, doc_02, doc_03",1,1.0,1.0,1.0
6,q07,как ячейку в экселе поменять если уже написал ...,doc_01,"doc_03, doc_01",1,1.0,0.5,2.0
7,q08,панель быстрого доступа в экселе где находится...,doc_01,"doc_01, doc_02",1,1.0,1.0,1.0
8,q09,можно ли в экселе строки столбцами поменять ме...,doc_01,doc_01,1,1.0,1.0,1.0


,metric,value
0,mean_hit@3,0.888889
1,mean_recall@3,0.888889
2,mean_MRR@3,0.833333


In [105]:
import logging
import warnings

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

chunk_configs = [
    {"chunk_size": 18, "overlap": 4},
    {"chunk_size": 28, "overlap": 8},
    {"chunk_size": 40, "overlap": 10},
    {"chunk_size": 60, "overlap": 15},
]

chunk_experiments = []

for cfg in chunk_configs:
    exp_artifacts = build_retriever(
        documents,
        chunk_size=cfg["chunk_size"],
        overlap=cfg["overlap"],
        device=DEVICE,
    )
    eval_df = evaluate_benchmark(benchmark_queries, artifacts=exp_artifacts, top_k=3)

    chunk_experiments.append(
        {
            "chunk_size": cfg["chunk_size"],
            "overlap": cfg["overlap"],
            "num_chunks": len(exp_artifacts.chunks_df),
            "backend_name": exp_artifacts.backend_name,
            "mean_hit@3": eval_df["hit@3"].mean(),
            "mean_recall@3": eval_df["recall@3"].mean(),
            "mean_MRR@3": eval_df["MRR@3"].mean(),
        }
    )

chunk_experiments_df = pd.DataFrame(chunk_experiments).sort_values(
    by=["mean_hit@3", "mean_MRR@3", "num_chunks"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(chunk_experiments_df)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5157.36it/s]


Используем dense-модель эмбеддингов.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7173.14it/s]


Используем dense-модель эмбеддингов.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5066.66it/s]


Используем dense-модель эмбеддингов.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5105.34it/s]


Используем dense-модель эмбеддингов.


,chunk_size,overlap,num_chunks,backend_name,mean_hit@3,mean_recall@3,mean_MRR@3
0,40,10,115,SentenceTransformer: sentence-transformers/par...,0.888889,0.888889,0.833333
1,18,4,246,SentenceTransformer: sentence-transformers/par...,0.888889,0.888889,0.777778
2,60,15,78,SentenceTransformer: sentence-transformers/par...,0.777778,0.777778,0.777778
3,28,8,172,SentenceTransformer: sentence-transformers/par...,0.777778,0.777778,0.777778


In [33]:
topk_rows = []

for top_k in [1, 2, 3, 4, 5]:
    eval_df = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=top_k)
    topk_rows.append(
        {
            "top_k": top_k,
            "mean_hit": eval_df[f"hit@{top_k}"].mean(),
            "mean_recall": eval_df[f"recall@{top_k}"].mean(),
            "mean_MRR": eval_df[f"MRR@{top_k}"].mean(),
        }
    )

topk_df = pd.DataFrame(topk_rows)
display(topk_df)

,top_k,mean_hit,mean_recall,mean_MRR
0,1,0.777778,0.777778,0.777778
1,2,0.888889,0.888889,0.833333
2,3,0.888889,0.888889,0.833333
3,4,0.888889,0.888889,0.833333
4,5,0.888889,0.888889,0.833333


### retrieval переиндексация

In [106]:
new_document = open_document("doc_04", "./upd_doc/руководство powerpoint.txt")

updated_documents = documents + [new_document]
print(updated_documents)

display(pd.DataFrame([new_document])[["doc_id", "title"]])

new_benchmark_queries: List[Dict[str, object]] = [{
        "query_id": "q10",
        "query": "в чем разница между pptx и ppsx в поверпоинте",
        "relevant_doc_ids": ["doc_04"],
    },
    {
        "query_id": "q11",
        "query": "как добавить анимацию на слайд в презентации",
        "relevant_doc_ids": ["doc_04"],
    }]
new_benchmark_queries_df = pd.DataFrame(new_benchmark_queries)
display(Markdown("### Как baseline-база отвечает на новые запросы"))
for query in new_benchmark_queries_df["query"]:
    display(Markdown(f"**Запрос:** {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

[{'doc_id': 'doc_01', 'title': 'руководство ecxel', 'text': "Панель быстрого доступа можно легко изменять и дополнять\nновыми командами. Для этого надо нажать кнопку Настройка панели\nбыстрого доступа, которая находится в ее правой части и представлена в виде стрелки, направленной вниз.\nЛинейки прокрутки\nЛинейки прокрутки (вертикальная и горизонтальная) (см. рис. 1.1)\nиспользуются для перемещения по содержимому документа. Чем\nбольше документ, тем меньше будет ползунок посередине линеек\nпрокрутки. Позиция ползунка позволяет определить, в каком месте\nдокумента сейчас находится пользователь – в начале, в конце или посередине.\nРабочая книга, рабочий лист, ярлыки листов\nФайл Microsoft Excel называется рабочей книгой. Рабочая книга\nсостоит из рабочих листов, имена которых (Лист1, Лист2, …) выведены на ярлыках в нижней части окна рабочей книги. Щелкая по ярлыкам, можно переходить от листа к листу внутри рабочей книги.\nДля прокручивания ярлыков используются кнопки слева от горизонтал

,doc_id,title
0,doc_04,руководство powerpoint


### Как baseline-база отвечает на новые запросы

**Запрос:** в чем разница между pptx и ppsx в поверпоинте

,rank,score,doc_id,title,chunk_text
0,1,0.332759,doc_02,Руководство Loginom,"прямоугольниками, у них справа. Компоненты экс..."
1,2,0.317956,doc_02,Руководство Loginom,"источников, приемников данных, к которым можно..."
2,3,0.301359,doc_02,Руководство Loginom,принимает одно из состояний: 1. Успешное (конт...


**Запрос:** как добавить анимацию на слайд в презентации

,rank,score,doc_id,title,chunk_text
0,1,0.563203,doc_01,руководство ecxel,на ярлыке Вставить лист в нижней части экрана ...
1,2,0.534725,doc_01,руководство ecxel,копируется Для доступа к другим способам надо ...
2,3,0.500790,doc_02,Руководство Loginom,отображается содержимое объекта пакета. Необхо...


In [107]:
updated_artifacts = build_retriever(
    updated_documents,
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

before_update_eval = evaluate_benchmark(new_benchmark_queries, artifacts=artifacts, top_k=3)
after_update_eval = evaluate_benchmark(new_benchmark_queries, artifacts=updated_artifacts, top_k=3)

comparison_df = before_update_eval.merge(
    after_update_eval,
    on=["query_id", "query", "relevant_doc_ids"],
    suffixes=("_before", "_after"),
)

display(comparison_df)

summary_comparison_df = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "before_update": [
            before_update_eval["hit@3"].mean(),
            before_update_eval["recall@3"].mean(),
            before_update_eval["MRR@3"].mean(),
        ],
        "after_update": [
            after_update_eval["hit@3"].mean(),
            after_update_eval["recall@3"].mean(),
            after_update_eval["MRR@3"].mean(),
        ],
    }
)
summary_comparison_df["delta"] = summary_comparison_df["after_update"] - summary_comparison_df["before_update"]
display(summary_comparison_df)

display(Markdown("### Как updated-база отвечает на новые запросы"))
for query in new_benchmark_queries_df["query"]:
    display(Markdown(f"**Запрос:** {query}"))
    display(search_chunks(query, artifacts=updated_artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5850.74it/s]


Используем dense-модель эмбеддингов.


,query_id,query,relevant_doc_ids,predicted_doc_ids_before,hit@3_before,recall@3_before,MRR@3_before,first_relevant_rank_before,predicted_doc_ids_after,hit@3_after,recall@3_after,MRR@3_after,first_relevant_rank_after
0,q10,в чем разница между pptx и ppsx в поверпоинте,doc_04,doc_02,0,0.0,0.0,None,"doc_04, doc_02",1,1.0,1.0,1
1,q11,как добавить анимацию на слайд в презентации,doc_04,"doc_01, doc_02",0,0.0,0.0,None,doc_04,1,1.0,1.0,1


,metric,before_update,after_update,delta
0,mean_hit@3,0.0,1.0,1.0
1,mean_recall@3,0.0,1.0,1.0
2,mean_MRR@3,0.0,1.0,1.0


### Как updated-база отвечает на новые запросы

**Запрос:** в чем разница между pptx и ppsx в поверпоинте

,rank,score,doc_id,title,chunk_text
0,1,0.395400,doc_04,руководство powerpoint,File Format). Расширение *.tif. Это полиграфич...
1,2,0.375022,doc_04,руководство powerpoint,презентации:  Презентация PowerPoint. Созданн...
2,3,0.332759,doc_02,Руководство Loginom,"прямоугольниками, у них справа. Компоненты экс..."


**Запрос:** как добавить анимацию на слайд в презентации

,rank,score,doc_id,title,chunk_text
0,1,0.790245,doc_04,руководство powerpoint,инструменты для добавления анимационных объект...
1,2,0.772824,doc_04,руководство powerpoint,демонстрации слайдов. Вкладка содержит команды...
2,3,0.745130,doc_04,руководство powerpoint,слайдов в презентации; панель Заметки служит д...


## RAG

In [66]:
import re
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def pick_best_sentences(query: str, text: str, top_n: int = 2) -> List[str]:
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentences).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    best_idx = np.argsort(-scores)[:top_n]
    return [sentences[i] for i in best_idx if scores[i] > 0]


def answer_without_retrieval(query: str, documents: List[Dict[str, str]]) -> Dict[str, object]:
    doc_texts = [doc["title"] + ". " + doc["text"] for doc in documents]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(doc_texts + [query]).toarray().astype(np.float32)

    doc_vecs = matrix[:-1]
    query_vec = matrix[-1]

    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-12
    query_norm = np.linalg.norm(query_vec) + 1e-12
    scores = (doc_vecs @ query_vec) / (doc_norms * query_norm)

    best_idx = int(np.argmax(scores))
    best_doc = documents[best_idx]
    best_sentences = pick_best_sentences(query, best_doc["text"], top_n=2)

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = (
            "Не удалось уверенно извлечь ответ без retrieval по чанкам. "
            "Система выбрала наиболее похожий документ целиком."
        )

    return {
        "answer": answer,
        "selected_doc_id": best_doc["doc_id"],
        "selected_title": best_doc["title"],
        "score": float(scores[best_idx]),
    }

In [67]:
baseline_example = answer_without_retrieval(
    "как ячейку в экселе поменять если уже написал туда текст",
    documents,
)

display(pd.DataFrame([baseline_example]))

,answer,selected_doc_id,selected_title,score
0,Даты до 1 января 1900 года воспринимаются как ...,doc_01,руководство ecxel,0.040722


In [75]:
# Базовые библиотеки для воспроизводимости, анализа и удобного вывода результатов.
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False
FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")
try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False
@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object


def build_retriever(
    documents: List[Dict[str, str]],
    chunk_size: int = 28,
    overlap: int = 8,
    device: str = "cpu",
) -> RetrieverArtifacts:
    rows = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f'{doc["doc_id"]}_chunk_{chunk_id:02d}',
                    "chunk_text": chunk_text_value,
                }
            )

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        backend=backend,
        index=index,
    )


def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)

    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores = scores[0]
        indices = indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

In [73]:
def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [76]:
query = "как в логиноме создать новый сценарий и узлы туда добавить"
context, retrieved_df = build_context_from_retrieval(query, artifacts=artifacts, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

### Запрос: как в логиноме создать новый сценарий и узлы туда добавить

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.666853,doc_02,Руководство Loginom,doc_02_chunk_27,перетащить мышкой в область построения сценари...
1,2,0.644301,doc_02,Руководство Loginom,doc_02_chunk_26,"в область построения, становится узлом. Област..."
2,3,0.589853,doc_02,Руководство Loginom,doc_02_chunk_24,"открытию в новых вкладках, все переходы происх..."


[Источник: doc_02 | Руководство Loginom | score=0.6669]
перетащить мышкой в область построения сценария). 2. Из контекстного меню нужного компонента выбрать команду Добавить узел в сценарий. Узлы в области построения сценария могут работать с несколькими видами объектов. Основные виды объектов: переменные и наборы данных. Наличие объектов узла зависит

[Источник: doc_02 | Руководство Loginom | score=0.6443]
в область построения, становится узлом. Область построения Сценария – полотно, содержащее узлы Сценария и связи между ними. 2 способа добавления узла в сценарий: 1. Используя Drag & Drop (нужный компонент перетащить мышкой в область построения сценария). 2. Из контекстного меню

[Источник: doc_02 | Руководство Loginom | score=0.5899]
открытию в новых вкладках, все переходы происходят в текущей вкладке. 1.2. Сценарий Сценарий – главная составная часть модуля, последовательность шагов по обработке данных. Шаги задаются узлами из компонентов: стандартными или производными. Стандартные 

In [77]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [78]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

Из контекстного меню нужного компонента выбрать команду Добавить узел в сценарий. Сценарий Сценарий – главная составная часть модуля, последовательность шагов по обработке данных.


In [79]:
def mini_rag_answer(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [81]:
rag_result = mini_rag_answer(
    "не могу найти куда в ворде нажать чтобы кнопку добавить на панель",
    artifacts=artifacts,
    top_k=3,
)

display(Markdown(f"### Вопрос: {rag_result['query']}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(Markdown("**Источники:**"))
display(rag_result["sources"])

### Вопрос: не могу найти куда в ворде нажать чтобы кнопку добавить на панель

**Ответ:** Если нажать на кнопку в конце При нажатии на кнопку Добавить или удалить кнопки появится меню (рис.6), в котором можно вывести или убрать кнопку с панели.

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.690070,doc_03,руководство word,doc_03_chunk_5,"команды, доступные в меню. Для вызова команды,..."
1,2,0.667720,doc_03,руководство word,doc_03_chunk_7,имя нужной панели. Если панель присутствует на...
2,3,0.663214,doc_03,руководство word,doc_03_chunk_8,которые были использованы последними. Если наж...


In [82]:
comparison_queries = [
    "Как редактировать ячейку в Excel после того как уже ввел данные?",
    "Как добавить кнопку на панель быстрого доступа в Word?",
    "Почему узел в Loginom становится красным при запуске?",
    "Чем отличается pptx от ppsx в PowerPoint?",
    "Как добавить анимацию на слайд в презентации?",
]

comparison_rows = []

for query in comparison_queries:
    baseline = answer_without_retrieval(query, documents)
    rag = mini_rag_answer(query, artifacts=artifacts, top_k=3)

    comparison_rows.append(
        {
            "query": query,
            "baseline_doc_id": baseline["selected_doc_id"],
            "baseline_score": baseline["score"],
            "baseline_answer": baseline["answer"],
            "rag_answer": rag["answer"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,query,baseline_doc_id,baseline_score,baseline_answer,rag_answer
0,Как редактировать ячейку в Excel после того ка...,doc_01,0.054429,"Ячейки строк вставляются как столбцы, ячейки с...",Microsoft Excel обычно распознает вводимые в я...
1,Как добавить кнопку на панель быстрого доступа...,doc_03,0.134254,"Чтобы добавить кнопку на\nпанель инструментов,...",Панель быстрого доступа можно легко изменять и...
2,Почему узел в Loginom становится красным при з...,doc_02,0.058836,Неуспешное (контур узла становится красным). В...,"о версии Loginom, редакции и др. При нажатии н..."
3,Чем отличается pptx от ppsx в PowerPoint?,doc_01,0.010287,"Чем\nбольше документ, тем меньше будет ползуно...",В найденном контексте нет достаточно релевантн...
4,Как добавить анимацию на слайд в презентации?,doc_03,0.061342,Отформатированные символы отображаются на\nэкр...,Сделать щелчок на ярлыке этого листа. на ярлык...


In [83]:
query = "Как сохранить презентацию в формате pdf"
rag_result = mini_rag_answer(query, artifacts=artifacts, top_k=3)

display(Markdown(f"### Вопрос: {query}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(rag_result["sources"][["rank", "score", "doc_id", "title", "chunk_text"]])

### Вопрос: Как сохранить презентацию в формате pdf

**Ответ:** документ в виде Web-страницы;  Разметка страниц – отображает документ в точном соответствии с тем, как он будет выведен на печать; в этом режиме удобно работать с колонтитулами, фреймами и многоколонной версткой документа; только в этом режиме отображается вертикальная координатная

,rank,score,doc_id,title,chunk_text
0,1,0.438459,doc_03,руководство word,многоколонной версткой документа; только в это...
1,2,0.434986,doc_03,руководство word,документ в виде Web-страницы;  Разметка стран...
2,3,0.386851,doc_01,руководство ecxel,направленной вниз. Линейки прокрутки Линейки п...


In [85]:
def keyword_recall(answer: str, expected_keywords: List[str]) -> float:
    answer_lower = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return hits / len(expected_keywords) if expected_keywords else np.nan


def evaluate_mini_rag(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []

    for item in benchmark_rows:
        query = item["query"]
        relevant_doc_ids = item["relevant_doc_ids"]
        expected_keywords = item["expected_keywords"]

        retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
        predicted_doc_ids = retrieved["doc_id"].tolist()
        retrieval_hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))

        baseline = answer_without_retrieval(query, documents)
        rag = mini_rag_answer(query, artifacts=artifacts, top_k=top_k)

        rows.append(
            {
                "query_id": item["query_id"],
                "query": query,
                "relevant_doc_ids": ", ".join(relevant_doc_ids),
                "predicted_doc_ids": ", ".join(predicted_doc_ids),
                f"retrieval_hit@{top_k}": retrieval_hit,
                "baseline_keyword_recall": keyword_recall(baseline["answer"], expected_keywords),
                "rag_keyword_recall": keyword_recall(rag["answer"], expected_keywords),
                "baseline_answer": baseline["answer"],
                "rag_answer": rag["answer"],
            }
        )

    return pd.DataFrame(rows)

In [87]:
qa_benchmark = [
    {
        "query_id": "q01",
        "query": "зачем нужен overlap когда режешь текст на куски",
        "relevant_doc_ids": ["doc_03"],
        "expected_keywords": ["перекрытие", "граница", "теряется", "контекст"],
    },
    {
        "query_id": "q02",
        "query": "как понять что поиск хорошо работает",
        "relevant_doc_ids": ["doc_04"],
        "expected_keywords": ["метрики", "hit", "recall", "mrr"],
    },
    {
        "query_id": "q03",
        "query": "добавил новые файлы в базу что дальше делать",
        "relevant_doc_ids": ["doc_05"],
        "expected_keywords": ["перестроить", "индекс", "заново", "эмбеддинги"],
    },
    {
        "query_id": "q04",
        "query": "когда лучше искать и по смыслу и по словам одновременно",
        "relevant_doc_ids": ["doc_10"],
        "expected_keywords": ["гибрид", "dense", "tf-idf", "лексика"],
    },
    {
        "query_id": "q05",
        "query": "зачем нужен второй этап поиска который пересортировывает результаты",
        "relevant_doc_ids": ["doc_09"],
        "expected_keywords": ["re-rank", "переранжировка", "уточняет", "кандидаты"],
    },
]

evaluation_df = evaluate_mini_rag(qa_benchmark, artifacts=artifacts, top_k=3)
display(evaluation_df)

,query_id,query,relevant_doc_ids,predicted_doc_ids,retrieval_hit@3,baseline_keyword_recall,rag_keyword_recall,baseline_answer,rag_answer
0,q01,зачем нужен overlap когда режешь текст на куски,doc_03,"doc_01, doc_03, doc_01",1,0.0,0.0,Режимы отображения документа\nРедактор Microso...,Для выделения нескольких столбцов или строк сл...
1,q02,как понять что поиск хорошо работает,doc_04,"doc_01, doc_01, doc_02",0,0.0,0.0,"Ячейки строк вставляются как столбцы, ячейки с...",В найденном контексте нет достаточно релевантн...
2,q03,добавил новые файлы в базу что дальше делать,doc_05,"doc_01, doc_03, doc_03",0,0.0,0.0,Новые столбцы всегда добавляются слева от выде...,В найденном контексте нет достаточно релевантн...
3,q04,когда лучше искать и по смыслу и по словам одн...,doc_10,"doc_03, doc_02, doc_01",0,0.0,0.0,По умолчанию он пустой. Вид по умолчанию – пли...,В группе Ячейки ленты Главная щелкнуть по стре...
4,q05,зачем нужен второй этап поиска который пересор...,doc_09,"doc_01, doc_01, doc_02",0,0.0,0.0,Второй способ. Первый способ – перетаскивание ...,В найденном контексте нет достаточно релевантн...


## Сохранение

In [90]:
os.makedirs("./artifacts", exist_ok=True)
save_path = "./artifacts"
retrieval_eval = pd.DataFrame({
    "query": baseline_eval_k3["query"],
    "expected_source": baseline_eval_k3["relevant_doc_ids"],
    "retrieved_sources": baseline_eval_k3["predicted_doc_ids"],
    "hit_at_k": baseline_eval_k3["hit@3"],
    "rank_of_first_relevant": baseline_eval_k3["first_relevant_rank"],
})

retrieval_eval.to_csv(
    "./artifacts/retrieval_eval.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: homeworks/HW14/artifacts/retrieval_eval.csv")
display(retrieval_eval)

Файл сохранён: homeworks/HW14/artifacts/retrieval_eval.csv


,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,а какие форматы у вордовских файлов? docx это ...,doc_03,"doc_03, doc_02",1,1.0
1,не могу найти куда в ворде нажать чтобы кнопку...,doc_03,doc_03,1,1.0
2,логином комьюнити это бесплатно? зачем оно воо...,doc_02,"doc_03, doc_01",0,NaN
3,как в логиноме создать новый сценарий и узлы т...,doc_02,doc_02,1,1.0
4,узел красным горит при запуске это ошибка? а з...,doc_02,"doc_02, doc_01",1,1.0
5,в экселе есть специальная вставка там кнопка т...,doc_01,"doc_01, doc_02, doc_03",1,1.0
6,как ячейку в экселе поменять если уже написал ...,doc_01,"doc_03, doc_01",1,2.0
7,панель быстрого доступа в экселе где находится...,doc_01,"doc_01, doc_02",1,1.0
8,можно ли в экселе строки столбцами поменять ме...,doc_01,doc_01,1,1.0


In [91]:
rag_examples_df = pd.DataFrame({
    "question": evaluation_df["query"],
    "answer": evaluation_df["rag_answer"],
    "retrieved_sources": evaluation_df["predicted_doc_ids"],
})

rag_examples_df.to_csv(
    "./artifacts/rag_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/rag_examples.csv")
display(rag_examples_df)

Файл сохранён: ./artifacts/rag_examples.csv


,question,answer,retrieved_sources
0,зачем нужен overlap когда режешь текст на куски,Для выделения нескольких столбцов или строк сл...,"doc_01, doc_03, doc_01"
1,как понять что поиск хорошо работает,В найденном контексте нет достаточно релевантн...,"doc_01, doc_01, doc_02"
2,добавил новые файлы в базу что дальше делать,В найденном контексте нет достаточно релевантн...,"doc_01, doc_03, doc_03"
3,когда лучше искать и по смыслу и по словам одн...,В группе Ячейки ленты Главная щелкнуть по стре...,"doc_03, doc_02, doc_01"
4,зачем нужен второй этап поиска который пересор...,В найденном контексте нет достаточно релевантн...,"doc_01, doc_01, doc_02"


In [108]:
print(comparison_df.columns.tolist())

['query_id', 'query', 'relevant_doc_ids', 'predicted_doc_ids_before', 'hit@3_before', 'recall@3_before', 'MRR@3_before', 'first_relevant_rank_before', 'predicted_doc_ids_after', 'hit@3_after', 'recall@3_after', 'MRR@3_after', 'first_relevant_rank_after']


In [109]:
retrieval_before_after_df = pd.DataFrame({
    "query": comparison_df["query"],
    "before_retrieved_sources": comparison_df["predicted_doc_ids_before"],
    "after_retrieved_sources": comparison_df["predicted_doc_ids_after"],
    "changed": (comparison_df["predicted_doc_ids_before"] != comparison_df["predicted_doc_ids_after"]),
})

retrieval_before_after_df.to_csv(
    "./artifacts/retrieval_before_after_update.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/retrieval_before_after_update.csv")
display(retrieval_before_after_df)

Файл сохранён: ./artifacts/retrieval_before_after_update.csv


,query,before_retrieved_sources,after_retrieved_sources,changed
0,в чем разница между pptx и ppsx в поверпоинте,doc_02,"doc_04, doc_02",True
1,как добавить анимацию на слайд в презентации,"doc_01, doc_02",doc_04,True
